In [14]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

from src.data_loader import load_data

In [16]:
data = load_data('data_2')

data['duration'] = (data.time_to - data.time_from).dt.total_seconds()

# Duration Prediction Part

## Feature Engineering

In [17]:
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureStore(BaseEstimator, TransformerMixin):
    def __init__(self): #, date_split):
      #  self.date_split = pd.to_datetime(date_split)
        self.ma_dictionary = {}

    def fit(self, X, y=None):
        df = X.copy()
        df['duration'] = (df['time_to'] - df['time_from']).dt.total_seconds()

        # Create dict for duration 7 days moving average with 1 day shift calculation for worker - item - process - date combo
        daily_stats = df.assign(date=df['time_from'].dt.normalize())\
            .groupby(['worker_id', 'item_id', 'process_name', 'date'], as_index=False)['duration']\
            .mean()
        daily_stats['ma_duration'] = daily_stats.sort_values('date')\
            .groupby(['worker_id', 'item_id', 'process_name'])['duration']\
            .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
        self.ma_dict = daily_stats.set_index(['worker_id', 'item_id', 'process_name', 'date'])['ma_duration'].to_dict()
        
        return self

    def transform(self, X):
        df = X.copy()
        
        # Create features from date
        df['weekday'] = df['time_from'].dt.weekday
        df['day'] = df['time_from'].dt.day
        df['date'] = df['time_from'].dt.normalize()
        
        # Map MA duration by worker - item - process - date combo
        df['ma_duration'] = df.apply(
            lambda row: self.ma_dict.get((row['worker_id'], row['item_id'], row['process_name'], row['date'])),
            axis=1
        )
        
        # Return only features
        categorical_features = ['worker_id', 'item_id', 'process_name'] 
        numerical_features = ['weekday', 'day', 'ma_duration']
        X_out = df[categorical_features + numerical_features].copy()

        X_out['ma_duration'] = X_out['ma_duration'].astype(float)

        # Change categorical feature type to category to pass it to LightGBM
        for f in categorical_features:
            X_out[f] = X_out[f].astype('category')
        
        return X_out

In [18]:
data['date'] = data.time_from.dt.normalize()
date_split = data['date'].max() - pd.Timedelta(days=7)

cv_metrics = []

for i in range(7):
    
    test_start = date_split + pd.Timedelta(days=i)
    test_end = test_start + pd.Timedelta(days=1)
    
    train_set = data[data['date'] < test_start]
    test_set = data[(data['date'] >= test_start) & (data['date'] < test_end)]

    print(f'Run {i} Train: {train_set.time_from.min(), train_set.time_from.max()} Test: {test_set.time_from.min(), test_set.time_from.max()}')
    
    if test_set.empty:
        continue
        
    fe_transformer = FeatureStore()
    
    X_train = fe_transformer.fit_transform(train_set)
    y_train = train_set['duration']
    
    X_test = fe_transformer.transform(test_set)
    y_test = test_set['duration']

    # TODO: tune hyperparameters, compare to train, add evaluation function
    model = LGBMRegressor(n_estimators=100, verbose=-1)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    print(f"Fold {i+1} Date: {test_start.date()} Test Samples: {len(test_set)} MAE: {mae:.2f} RMSE: {rmse:.2f} R2 score: {r2:.2f}")
    cv_metrics.append(mae)

print(f"Average MAE {np.mean(cv_metrics):.2f}")

Run 0 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-09-22 22:43:32')) Test: (Timestamp('2025-09-23 08:08:11'), Timestamp('2025-09-23 22:56:59'))
Fold 1 Date: 2025-09-23 Test Samples: 41 MAE: 5.10 RMSE: 6.44 R2 score: 0.57
Run 1 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-09-23 22:56:59')) Test: (Timestamp('2025-09-24 08:01:36'), Timestamp('2025-09-24 22:47:42'))
Fold 2 Date: 2025-09-24 Test Samples: 45 MAE: 6.98 RMSE: 8.73 R2 score: 0.17
Run 2 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-09-24 22:47:42')) Test: (Timestamp('2025-09-25 08:42:14'), Timestamp('2025-09-25 22:43:54'))
Fold 3 Date: 2025-09-25 Test Samples: 53 MAE: 5.55 RMSE: 7.10 R2 score: 0.65
Run 3 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-09-25 22:43:54')) Test: (Timestamp('2025-09-26 08:26:31'), Timestamp('2025-09-26 22:59:09'))
Fold 4 Date: 2025-09-26 Test Samples: 28 MAE: 5.57 RMSE: 6.58 R2 score: 0.67
Run 4 Train: (Timestamp('2025-07-01 08:06:01'), Timestamp('2025-

In [27]:
# TODO: Remove the last day from training
final_fe_transformer = FeatureStore()
X_final_train = final_fe_transformer.fit_transform(data)
y_final_train = data['duration']

final_model = LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
final_model.fit(X_final_train, y_final_train)

pipe = Pipeline([
    ('feature_engineering', final_fe_transformer),
    ('lightgbm_model', final_model)
])

MODEL_PATH = PROJECT_ROOT / "src" / "task2" / "duration_fcst_pipe.pkl"

with MODEL_PATH.open("wb") as f:
    pickle.dump(pipe, f)